In [1]:
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 

import numpy as np 
import tensorflow as tf 
from tensorflow.keras import * 

from tensorflow.keras.layers import * 
from tensorflow.keras.models import load_model, save_model 
from tensorflow.keras import Model
import tensorflow.keras.utils as utils
from keras.datasets import mnist 


### Original Floating-point Model 

In [49]:
def Net(): 
    # inputs = Input(shape=x_train.shape[1:])
    inputs = Input(shape=(28, 28, 1))  
    x = Conv2D(filters=4, kernel_size=(3,3), strides=(1,1), padding='same')(inputs)
    x = ReLU()(x)
    x = MaxPool2D((2,2),strides=(2,2), padding="same")(x)

    x = Flatten()(x)
    x = Dense(20)(x)

    x = ReLU()(x)
    x = Dense(10)(x)
    predictions = Softmax()(x)
    return Model(inputs=inputs, outputs=predictions)

In [50]:
epochs = 5 
num_classes = 10 
(x_train, y_train_num), (x_test, y_test_num) = mnist.load_data() 
x_train = x_train.reshape(x_train.shape[0], x_train.shape[1], x_train.shape[2], 1)
x_test = x_test.reshape(x_test.shape[0], x_test.shape[1], x_test.shape[2], 1)
y_train = utils.to_categorical(y_train_num, num_classes)
y_test = utils.to_categorical(y_test_num, num_classes)
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

model = Net()
model.load_weights('neural_network_model.h5')

(60000, 28, 28, 1) (10000, 28, 28, 1) (60000, 10) (10000, 10)


In [51]:
length = y_test.shape[0]
positives = 0
res = 0 
for i in range(length): 
    pred = model(x_test[i:i+1]).numpy().squeeze()
    y_test = (y_test > 0.5)
    if i == 5 or i == 10 or i == 15: 
        print("y_test :", y_test[i])
        print("pred   :", pred)
        print("res    :", y_test[i]==pred)
    pred = (pred > 0.5)
    positives = np.sum(y_test[i] == pred)
    if positives == 10: 
        res += 1
        
print("Accuracy:", res/length)

y_test : [False  True False False False False False False False False]
pred   : [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
res    : [ True  True  True  True  True  True  True  True  True  True]
y_test : [ True False False False False False False False False False]
pred   : [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
res    : [ True  True  True  True  True  True  True  True  True  True]
y_test : [False False False False False  True False False False False]
pred   : [0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
res    : [ True  True  True  True  True  True  True  True  True  True]
Accuracy: 0.885


In [38]:
"""
quantize numpy array
return:
q_x: quantized array
fp_x: reverse quantized array to float
"""
def quantize_array(x, bit_depth=8):
    min_x = x.min() 
    max_x = x.max()

    #find number of integer bits to represent this range
    int_bits = int(np.ceil(np.log2(max(abs(min_x),abs(max_x)))))
    
    if int_bits < 0: int_bits = 0

    frac_bits = bit_depth - 1 - int_bits #remaining bits are fractional bits (1-bit for sign)

    #floating point weights are scaled and rounded to [-2^(bit-1),2^(bit-1)-1], which are used in 
    #the fixed-point operations on the actual hardware (i.e., microcontroller)
    q_x = np.round(x*(2**frac_bits))

    # To quantify the impact of quantized weights, scale them back to
    # original range to run inference using quantized weights
    fp_x = q_x/(2**frac_bits)
    
    return q_x, fp_x, int_bits, frac_bits


"""
compute frac_bits based on max and min list
used for activation(feature map) quantization
"""
def min_max_quantize(max_list, min_list, bit_depth=8):
    frac_list = []
    for i in range(len(max_list)):
        int_bits = int(np.ceil(np.log2(max(abs(min_list[i]),abs(max_list[i])))))
        if int_bits < 0: int_bits = 0
        frac_bits = bit_depth - 1 - int_bits #remaining bits are fractional bits (1-bit for sign)
        frac_list.append(frac_bits)
    return frac_list

### Quantized Model

In [59]:
def QNet(bit=32, shift_list=[]):
    shift = False 
    if bit != 32: 
        shift = True 
    
    input = Input(shape=x_train.shape[1:], name='input')
    x = Conv2D(filters=4, kernel_size=(3,3), strides=(1,1), padding='same', name='conv1')(input)
    if shift:
        x = x / 2**shift_list[0]
        x = tf.floor(x)
        x = tf.clip_by_value(x, -2**(bit-1), 2**(bit-1)-1)  
    x = ReLU()(x)

    x = MaxPool2D((2,2), strides=(2,2), padding='same', name='max_pool')(x)
    x = Flatten()(x)

    x = Dense(20, name='fc1')(x)
    if shift:
        x = x / 2**shift_list[1]
        x = tf.floor(x)
        x = tf.clip_by_value(x, -2**(bit-1), 2**(bit-1)-1)
    x = ReLU()(x)
    
    x = Dense(10, name='fc2')(x)

    if shift: 
        x = x / 2**shift_list[2]
        x = tf.floor(x)
        x = tf.clip_by_value(x, -2**(bit-1), 2**(bit-1)-1)
    if not shift:
        x_out = Softmax()(x)
    else:
        x_out = x
    
    return Model(input, x_out)

model = QNet()
model.load_weights('neural_network_model.h5')

layer_names = [] 
for layer in model.layers: 
    if len(layer.get_weights()) != 0:
        layer_names.append(layer.name)
print("Layer names: ", layer_names)
layer_num = len(layer_names)

Layer names:  ['conv1', 'fc1', 'fc2']


In [60]:
# intermediate feature map retrieval 
# for max and min value acquisition and computing quantization format

intermediate_output = [model.get_layer('input').output]
for i in range(layer_num):
    intermediate_output.append(model.get_layer(layer_names[i]).output)
    
intermediate_layer_model = Model(inputs = model.input, outputs = intermediate_output)

frac_bits_output_list = [] 
max_list = []
min_list = [] 
for i in range(length):
    pred = intermediate_layer_model(x_test[i:i+1])
    for i in range(len(pred)):
        pred_np = pred[i].numpy()
        max_np = pred_np.max()
        min_np = pred_np.min()
        if len(max_list) != len(pred):
            max_list.append(max_np)
            min_list.append(min_np)
        else:
            if max_np > max_list[i]:
                max_list[i] = max_np
            if min_np < min_list[i]:
                min_list[i] = min_np

bit_depth = 16 
frac_bits_activation_list = min_max_quantize(max_list, min_list, bit_depth=bit_depth)

print("input: ", frac_bits_activation_list[0])
for i in range(layer_num):
    print(i+1, " ", layer_names[i], " ", frac_bits_activation_list[i+1])


input:  7
1   conv1   5
2   fc1   2
3   fc2   2


### Quantization


In [61]:
bit_depth = 16 
frac_bits_weight_list = [] 
frac_bits_bias_list = [] 
q_weight_list = [] 
q_bias_list = [] 

for i in range(layer_num):
    weight, bias = model.get_layer(layer_names[i]).get_weights()
    q_weight, f_weight, int_bits_weight, frac_bits_weight = quantize_array(weight, bit_depth=bit_depth)
    q_bias, f_bias, int_bits_bias, frac_bits_bias = quantize_array(bias, bit_depth=bit_depth)
    
    q_weight_list.append(q_weight)
    q_bias_list.append(q_bias)
    frac_bits_weight_list.append(frac_bits_weight)
    frac_bits_bias_list.append(frac_bits_bias)
    print('***********')
    print(layer_names[i] + '-- weight -- Q' + str(int_bits_weight) + '.' + str(frac_bits_weight))
    print(layer_names[i] + '-- bias -- Q' + str(int_bits_bias) + '.' + str(frac_bits_bias))    


***********
conv1-- weight -- Q1.14
conv1-- bias -- Q0.15
***********
fc1-- weight -- Q0.15
fc1-- bias -- Q0.15
***********
fc2-- weight -- Q0.15
fc2-- bias -- Q0.15


In [62]:
# computing bias shift-left and output shift-right
bias_shift_list = np.array(frac_bits_weight_list) + np.array(frac_bits_activation_list[0:-1]) - np.array(frac_bits_bias_list)
output_shift_list = np.array(frac_bits_weight_list) + np.array(frac_bits_activation_list[0:-1]) - np.array(frac_bits_activation_list[1:])  
print(bias_shift_list)
print(output_shift_list)

[6 5 2]
[16 18 15]


### Saving quantized weight

In [63]:
reordered_weight_list = []
for i in range(len(q_weight_list)):
    # convolution layer
    if len(q_weight_list[i].shape) == 4:
        reordered_weight_list.append(np.moveaxis(q_weight_list[i], 2, 0))
    # dense layer
    else: 
        reordered_weight_list.append(np.moveaxis(q_weight_list[i], 1, 0))

# flatten the weight tensor 
reordered_weight_flatten_list = []
for i in range(len(q_weight_list)): 
    # convolution layer 
    if len(reordered_weight_list[i].shape) == 4:
        reordered_weight_flatten_list.append(reordered_weight_list[i].flatten('F'))
    # dense layer
    else: 
        reordered_weight_flatten_list.append(reordered_weight_list[i].flatten())

# save 
f = open('./quantized_weight.h', 'w')
for i in range(len(q_weight_list)): 
    print('********' + layer_names[i] + '********')
    print('Bias shift: ', bias_shift_list[i])
    print('Output shift: ', output_shift_list[i])
    print('Bias: ', q_bias_list[i].astype(np.int16))
    print('Weight: ', reordered_weight_flatten_list[i].astype(np.int16))
    
    f.write('#define ' + layer_names[i].upper() + '_WT {' + str(reordered_weight_flatten_list[i].astype(np.int16).tolist())[1:-1] + '}\n')
    f.write('#define ' + layer_names[i].upper() + '_BIAS {' + str(q_bias_list[i].astype(np.int16).tolist())[1:-1] + "}\n")
    f.write('#define ' + layer_names[i].upper() + '_BIAS_LSHIFT ' + str(bias_shift_list[i]) + "\n")
    f.write('#define ' + layer_names[i].upper() + '_OUT_RSHIFT ' + str(output_shift_list[i]) + "\n\n")
    
f.close()

********conv1********
Bias shift:  6
Output shift:  16
Bias:  [2201 1520  207 2458]
Weight:  [  8406   9357  -2454   4706   9241   3335   -397  10323   1326 -14350
  -2323   6958   2221   5810   7165   2512  11337   1327   6274    272
   2981   9652   6987   1289   7059  -3666   5606   1972 -11224 -17284
     11  -3605 -12717  15139  14448   9648]
********fc1********
Bias shift:  5
Output shift:  18
Bias:  [ 4317 -3337  -210  -304  -182 -3250 -1273  5871  5256 -2090  2171   209
  -403 -4364  4434  4922 -2861 -1042 -2744  2424]
Weight:  [4022 4356 1976 ... 5535 3545 5421]
********fc2********
Bias shift:  2
Output shift:  15
Bias:  [ -590  6638 -1385  3822 -4798  3286  1777  2647 -9010   270]
Weight:  [  7678  -9049  18223   1920  -3680 -11178  -8527 -13172 -14892   2144
 -14143   5668  11519  -5586   3723 -21161  -2009  17637  -3384   9497
  12074 -10996 -17808  12118  10990  -3126 -16310   6093 -20622   3895
   6323  16650  -9424   4097  16304  12577 -18344  -5193   2306   7912
  11850

In [64]:
q_model = QNet(bit=16, shift_list=output_shift_list)

for i in range(layer_num):
    q_model.get_layer(layer_names[i]).set_weights([q_weight_list[i], q_bias_list[i] * (2**bias_shift_list[i])])

# evaluation 
length = y_test.shape[0]
positives = 0
for i in range(length):
    pred = q_model(np.round(x_test[i:i+1]*(2**frac_bits_activation_list[0]))).numpy().squeeze()
    pred = pred > 0.5
    print(i, " y_test : ", y_test[i])
    print(i, " pred   : ", pred)
    print(i, " result : ", y_test[i]==pred)
    positives += np.sum(y_test[i] == pred)
    
print("Accuracy after quantization: ", positives / length)

0  y_test :  [False False False False False False False  True False False]
0  pred   :  [False False  True  True False False False  True  True  True]
0  result :  [ True  True False False  True  True  True  True False False]
1  y_test :  [False False  True False False False False False False False]
1  pred   :  [False  True  True  True False  True False False  True False]
1  result :  [ True False  True False  True False  True  True False  True]
2  y_test :  [False  True False False False False False False False False]
2  pred   :  [False  True  True  True False False False False  True False]
2  result :  [ True  True False False  True  True  True  True False  True]
3  y_test :  [ True False False False False False False False False False]
3  pred   :  [ True False False False False  True  True False  True  True]
3  result :  [ True  True  True  True  True False False  True False False]
4  y_test :  [False False False False  True False False False False False]
4  pred   :  [False False